In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import altair as alt

In [ ]:
# Configuración del estilo visual
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12,8) # tamaño de la figura
# Deshabilitar el limite de 5000 filas de Altair
alt.data_transformers.enable("default", max_rows=None)
print("Librerías importadas")

In [ ]:
# carga y limpieza de datos
url = "https://data.insideairbnb.com/mexico/df/mexico-city/2026-06-15/data/listings.csv.gz"
print("Dataset cargado")
try:
  df = pl.read_csv(url)
  print(f"Dataset cargado exitosamente")
except Exception as e:
  print(f"Error al cargar el data set: {e}")
# limpieza inicial
if not df.is_empty():
  df= df.with_columns(
      pl.col("price")
      .str.replace_all(r"\$","")
      .str.replace_all(r",","")
      .cast(pl.Float64,strict=False)
      .alias("price")
      ).drop_nulls(
          subset=["price"]
      ).filter(
          pl.col("price")>0
      )
print("Limpieza inicial realizada")
display(df.head())


In [ ]:
# análisis básico
## mias
print("1. Cuál es el precio promedio por noche?")
avg_price = df.select(pl.col("price").mean()).item()
## avg_price2 = df["price"].mean()
display(avg_price)

print("2. Cuáles son los tipos de alojamiento?")
room_types = df.group_by("room_type").agg(pl.len().alias("count")).sort("count",descending=True)
## room_type2 = df.group_by("room_type").count()
display(room_types)

print("3. Cuáles son las 10 alcaldías con más alojamientos?")
neighborhoods_top10 = df.group_by("neighbourhood_cleansed").agg(pl.len().alias("count")).sort("count",descending=True).head(16)
## neighborhoods2 = df.group_by("neighbourhood_cleansed").count().sort("count",descending=True).head(10)
display(neighborhoods_top10)

print("4. ¿Quienes son los anfitriones con más alojamientos?")
host_names = df.group_by("host_name").agg(pl.len().alias("count")).sort("count",descending=True).head(10)
## host_names2 = df["host_name"].value_counts().sort("count",descending=True).head(10)
display(host_names)


In [ ]:
# Visualizaciones
price_to_plot_df = df.filter(pl.col("price") < df.select(pl.col("price").quantile(0.95)).item()).to_pandas()

# Usando altair
chart_hist = alt.Chart(price_to_plot_df).mark_bar().encode(
    alt.X("price:Q", bin=alt.Bin(maxbins=50), title="Precio por noche"),
    alt.Y("count()", title="Frecuencia"),
    tooltip=[alt.Tooltip("count()",title="Frecuencia"), alt.Tooltip("price:Q",bin=True,title="Rango de precio")]
    ).properties(
    title="Distribución de precios por noche",
    width=700,
    height=400
)

chart_hist.show()

In [ ]:
top_alcaldias_pd = neighborhoods_top10.to_pandas()

fig_hoods =px.bar(top_alcaldias_pd,
                  x="count",
                  y="neighbourhood_cleansed",
                  title="Alcaldías con alojamientos",
                  orientation="h",
                  labels={"count":"Numero de alojamientos", "neighbourhood_cleansed":"Alcaldía"},
                  color="neighbourhood_cleansed",
                  color_continuous_scale= px.colors.sequential.Viridis)
fig_hoods.show()

COMENTADO POR TEMAS DE ESPACIO

In [ ]:
#Gráfico 5: Mapà Geográfico de Precios
df_sample_for_plot = df. filter(pl.col('price') < df. select(pl. col('price').quantile(0.95)).item()).sample(n=5000,seed=42).to_pandas()
# Versión con Plotly Express (Mapa Interactivo)
# Nota: Plotly usa Mapbox para los mapas.
fig_map = px.scatter_mapbox(
  df_sample_for_plot,
  lat="latitude",
  lon="longitude",
  color="price",
  size="price",
  color_continuous_scale=px.colors.sequential.Viridis_r,
  size_max=15,
  zoom=10,
  mapbox_style="carto-positron",
  hover_name="name",
  hover_data={"neighbourhood_cleansed": True, "price": ":$.2f"})
fig_map.update_layout(
  title='Mapa Interactivo de Precios de Airbnb en CDMX', legend_title_text="Precio (MXN)")
fig_map.show()